# Práctica 6 · Agrupamiento de documentos
### Comparación de TF-IDF y CountVectorizer

**Objetivo:** comprobar cómo cambia K-means al representar los mismos textos mediante TF-IDF o frecuencias de palabras (BoW).

Usaremos **20 Newsgroups**, cuatro categorías y los parámetros de la *Sesión 19* (diapositivas 40–48). Las categorías reales se utilizarán **solo para evaluar**, no para entrenar los clusters.

> Ejecuta las celdas de arriba hacia abajo. La primera descarga de 20 Newsgroups requiere internet (por ejemplo, en Google Colab).

## 1. Librerías y datos

Trabajaremos con `scikit-learn`, `pandas`, `numpy` y `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import TruncatedSVD

pd.set_option("display.max_colwidth", 120)

In [ ]:
categorias = [
    "sci.space",
    "rec.sport.baseball",
    "comp.graphics",
    "talk.politics.mideast",
]

datos = fetch_20newsgroups(
    subset="train",
    categories=categorias,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=42,
)

documentos = datos.data
print(f"Documentos: {len(documentos)}")
print("Ejemplo de texto:\n", documentos[11][:500])

## 2. Representaciones vectoriales

- **CountVectorizer:** cuenta las apariciones de cada término.
- **TF-IDF:** reduce el peso relativo de términos frecuentes en muchos documentos.

Utilizaremos los mismos filtros en ambos métodos. `TfidfVectorizer` normaliza automáticamente con `norm="l2"`; `CountVectorizer` necesita normalización aparte.

*Nota:* el TF-IDF de scikit-learn utiliza por defecto IDF suavizado, diferente de la fórmula simplificada vista en algunas diapositivas.

In [ ]:
parametros = dict(
    stop_words="english",
    min_df=5,
    max_df=0.8,
    max_features=5000,
)

vectorizador_tfidf = TfidfVectorizer(**parametros, norm="l2")
X_tfidf = vectorizador_tfidf.fit_transform(documentos)

vectorizador_count = CountVectorizer(**parametros)
X_count_original = vectorizador_count.fit_transform(documentos)
X_count = normalize(X_count_original, norm="l2")

print("TF-IDF:", X_tfidf.shape)
print("CountVectorizer:", X_count.shape)
print("Mismo vocabulario:", np.array_equal(
    vectorizador_tfidf.get_feature_names_out(),
    vectorizador_count.get_feature_names_out()
))

### Comprobación de la norma L2

Para cada documento no vacío, la norma debe ser aproximadamente 1:

\[
\hat{x}=\frac{x}{\|x\|_2},\qquad \|\hat{x}\|_2=1.
\]

Los textos vacíos después del filtrado conservan un vector cero.

In [ ]:
def normas_filas(X):
    return np.sqrt(X.multiply(X).sum(axis=1)).A1

normas_antes = normas_filas(X_count_original)
normas_count = normas_filas(X_count)
normas_tfidf = normas_filas(X_tfidf)

for nombre, valores in [("Count + L2", normas_count), ("TF-IDF", normas_tfidf)]:
    no_nulos = valores > 0
    print(f"{nombre}: {no_nulos.sum()} no nulos; ", end="")
    print("norma≈1:", np.allclose(valores[no_nulos], 1.0))

plt.figure(figsize=(7, 3.5))
plt.hist(normas_antes, bins=40)
plt.xlabel("Norma L2 antes de normalizar")
plt.ylabel("Número de documentos")
plt.title("Longitud de los documentos: CountVectorizer")
plt.tight_layout()
plt.show()

## 3. K-means para K = 2, ..., 8

Para cada representación calculamos:

- **Inercia:** dispersión dentro de los clusters; menor suele ser mejor, pero siempre baja al aumentar K.
- **Silueta:** cohesión y separación; mayor suele ser mejor.

Seguimos las diapositivas: `random_state=42`, `n_init=10`, distancia euclidiana y una muestra de hasta 1000 documentos para la silueta.

In [ ]:
valores_k = range(2, 9)


def evaluar_kmeans(X, representacion):
    modelos = {}
    filas = []

    for k in valores_k:
        modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
        etiquetas = modelo.fit_predict(X)
        silueta = silhouette_score(
            X, etiquetas, metric="euclidean",
            sample_size=min(1000, X.shape[0]), random_state=42,
        )
        modelos[k] = modelo
        filas.append({
            "Representación": representacion,
            "K": k,
            "Inercia": modelo.inertia_,
            "Silueta": silueta,
        })

    return pd.DataFrame(filas), modelos


resultados_tfidf, modelos_tfidf = evaluar_kmeans(X_tfidf, "TF-IDF")
resultados_count, modelos_count = evaluar_kmeans(X_count, "Count + L2")
resultados = pd.concat([resultados_tfidf, resultados_count], ignore_index=True)

display(resultados.round(4))

## 4. Visualización: inercia y silueta

**Atención:** la inercia solo se compara entre distintos valores de K **dentro de la misma representación**, no directamente entre TF-IDF y CountVectorizer.

In [ ]:
# Curvas de inercia, una gráfica por representación.
for nombre, tabla in [("TF-IDF", resultados_tfidf), ("Count + L2", resultados_count)]:
    plt.figure(figsize=(6.5, 3.5))
    plt.plot(tabla["K"], tabla["Inercia"], marker="o")
    plt.xticks(list(valores_k))
    plt.xlabel("Número de clusters (K)")
    plt.ylabel("Inercia")
    plt.title(f"Método del codo — {nombre}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# La silueta sí permite una comparación descriptiva entre las representaciones.
plt.figure(figsize=(7, 4))
plt.plot(resultados_tfidf["K"], resultados_tfidf["Silueta"], "o-", label="TF-IDF")
plt.plot(resultados_count["K"], resultados_count["Silueta"], "s-", label="Count + L2")
plt.xticks(list(valores_k))
plt.xlabel("Número de clusters (K)")
plt.ylabel("Silueta promedio")
plt.title("Comparación de silueta")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Elegir K y comparar los resultados

Usaremos **K = 4** como referencia para comparar ambas representaciones con el **mismo número de grupos** (el conjunto seleccionado trata cuatro grandes temas). También informamos el K con mayor silueta, que puede ser diferente.

Puedes modificar `k_elegido` después de revisar las gráficas.

In [ ]:
k_elegido = 4

filas_resumen = []
for nombre, tabla in [("TF-IDF", resultados_tfidf), ("Count + L2", resultados_count)]:
    mejor = tabla.loc[tabla["Silueta"].idxmax()]
    elegida = tabla.loc[tabla["K"] == k_elegido].iloc[0]
    filas_resumen.append({
        "Representación": nombre,
        "K elegido": k_elegido,
        "Silueta (K elegido)": elegida["Silueta"],
        "K con mejor silueta": int(mejor["K"]),
        "Mejor silueta": mejor["Silueta"],
    })

resumen = pd.DataFrame(filas_resumen)
display(resumen.round(4))

modelos_elegidos = {
    "TF-IDF": modelos_tfidf[k_elegido],
    "Count + L2": modelos_count[k_elegido],
}

## 6. Interpretar los clusters

Extraemos las **10 palabras con mayor peso en cada centroide**. Los números de cluster son arbitrarios: el `cluster 0` de un método no tiene por qué corresponder al `cluster 0` del otro.

In [ ]:
def palabras_principales(modelo, vectorizador, cantidad=10):
    terminos = vectorizador.get_feature_names_out()
    filas = []
    for i, centroide in enumerate(modelo.cluster_centers_):
        indices = np.argsort(centroide)[-cantidad:][::-1]
        filas.append({
            "Cluster": i,
            "10 palabras principales": ", ".join(terminos[indices]),
        })
    return pd.DataFrame(filas)


palabras_tfidf = palabras_principales(modelos_elegidos["TF-IDF"], vectorizador_tfidf)
palabras_count = palabras_principales(modelos_elegidos["Count + L2"], vectorizador_count)

print("TF-IDF")
display(palabras_tfidf)
print("CountVectorizer + L2")
display(palabras_count)

In [ ]:
# Barras: peso de las 10 palabras principales de cada centroide.
for nombre, vectorizador in [("TF-IDF", vectorizador_tfidf), ("Count + L2", vectorizador_count)]:
    modelo = modelos_elegidos[nombre]
    terminos = vectorizador.get_feature_names_out()
    fig, ejes = plt.subplots(k_elegido, 1, figsize=(9, 2.7 * k_elegido))
    ejes = np.atleast_1d(ejes)

    for i, eje in enumerate(ejes):
        centroide = modelo.cluster_centers_[i]
        indices = np.argsort(centroide)[-10:]
        eje.barh(terminos[indices], centroide[indices])
        eje.set_title(f"{nombre} · Cluster {i}")
        eje.set_xlabel("Peso en el centroide")

    fig.tight_layout()
    plt.show()

## 7. Comparación con las categorías reales

Las categorías no se utilizaron en el entrenamiento. Ahora las recuperamos **solo para analizar** qué temas aparecen en cada cluster.

In [ ]:
etiquetas_reales = [datos.target_names[i] for i in datos.target]

tablas = {}
for nombre, modelo in modelos_elegidos.items():
    tabla = pd.crosstab(
        pd.Series(modelo.labels_, name="Cluster"),
        pd.Series(etiquetas_reales, name="Categoría real"),
    )
    tablas[nombre] = tabla
    print(f"{nombre} — cantidad de documentos por categoría:")
    display(tabla)

In [ ]:
# Cada fila suma 100 %: muestra la composición temática del cluster.
for nombre, tabla in tablas.items():
    porcentajes = tabla.div(tabla.sum(axis=1), axis=0) * 100

    fig, eje = plt.subplots(figsize=(8.5, 3.5 + 0.3 * k_elegido))
    imagen = eje.imshow(porcentajes, vmin=0, vmax=100, cmap="Blues", aspect="auto")
    eje.set_xticks(range(len(tabla.columns)), labels=tabla.columns, rotation=25, ha="right")
    eje.set_yticks(range(len(tabla.index)), labels=[f"Cluster {i}" for i in tabla.index])
    eje.set_title(f"Categorías originales por cluster — {nombre}")

    for i in range(porcentajes.shape[0]):
        for j in range(porcentajes.shape[1]):
            eje.text(j, i, f"{porcentajes.iloc[i, j]:.0f}%", ha="center", va="center",
                     color="white" if porcentajes.iloc[i, j] > 50 else "black")

    fig.colorbar(imagen, ax=eje, label="Porcentaje dentro del cluster")
    fig.tight_layout()
    plt.show()

## 8. Visualización en dos dimensiones (SVD)

Reducimos a dos dimensiones **solo para visualizar**. K-means fue entrenado con todos los términos, no con las coordenadas 2D.

In [ ]:
matrices = {"TF-IDF": X_tfidf, "Count + L2": X_count}

for nombre, X in matrices.items():
    coordenadas = TruncatedSVD(n_components=2, random_state=42).fit_transform(X)
    grupos = modelos_elegidos[nombre].labels_

    plt.figure(figsize=(7, 4.5))
    plt.scatter(coordenadas[:, 0], coordenadas[:, 1], c=grupos, cmap="tab10",
                s=10, alpha=0.6)
    plt.xlabel("Componente SVD 1")
    plt.ylabel("Componente SVD 2")
    plt.title(f"Proyección de documentos: {nombre} (K={k_elegido})")
    plt.tight_layout()
    plt.show()

## 9. Preguntas de análisis

Las respuestas siguientes se generan parcialmente a partir de los resultados del notebook. **Revisa las palabras y las tablas antes de entregar**, especialmente las interpretaciones de los temas.

**1. ¿Cuál es la diferencia entre TF-IDF y CountVectorizer?**  
CountVectorizer utiliza frecuencias directas; TF-IDF reduce el peso relativo de términos frecuentes en muchos documentos.

**2. ¿Por qué normalizamos CountVectorizer?**  
Para evitar que la magnitud del vector, asociada en parte a la longitud del documento, domine las distancias euclidianas. Cada vector no nulo queda con norma L2 igual a 1.

In [ ]:
# Respuestas basadas en los resultados calculados. Complementar la interpretación personal.
mejor_t = resultados_tfidf.loc[resultados_tfidf["Silueta"].idxmax()]
mejor_c = resultados_count.loc[resultados_count["Silueta"].idxmax()]

# Pureza descriptiva: proporción de documentos pertenecientes a la categoría
# más numerosa dentro de cada cluster (solo para evaluar, no para entrenar).
pureza = {
    nombre: tabla.max(axis=1).sum() / tabla.to_numpy().sum()
    for nombre, tabla in tablas.items()
}

coincidencia = adjusted_rand_score(
    modelos_elegidos["TF-IDF"].labels_, modelos_elegidos["Count + L2"].labels_
)

pal_t = palabras_tfidf.iloc[0]["10 palabras principales"].split(", ")[:5]
pal_c = palabras_count.iloc[0]["10 palabras principales"].split(", ")[:5]

texto = f"""
**3. ¿Qué K tuvo la mayor silueta?**  
TF-IDF: **K={int(mejor_t['K'])}**, silueta **{mejor_t['Silueta']:.4f}**.  
Count + L2: **K={int(mejor_c['K'])}**, silueta **{mejor_c['Silueta']:.4f}**.

**4. ¿Con qué representación son más fáciles de interpretar los clusters?**  
Para **K={k_elegido}**, observa las palabras principales de cada grupo. Por ejemplo, el primer cluster de TF-IDF incluye **{', '.join(pal_t)}**; el primer cluster de Count + L2 incluye **{', '.join(pal_c)}**.  
Pureza descriptiva: TF-IDF **{pureza['TF-IDF']:.1%}**, Count + L2 **{pureza['Count + L2']:.1%}**. La pureza puede orientar la comparación, pero **la facilidad de interpretación debe justificarse inspeccionando las palabras de todos los clusters**.

**5. ¿Los clusters coinciden con las categorías originales?**  
Revisa las tablas de contingencia. Con K={k_elegido}, las purezas son **{pureza['TF-IDF']:.1%}** (TF-IDF) y **{pureza['Count + L2']:.1%}** (Count + L2). Las asignaciones de ambas representaciones tienen un índice Rand ajustado de **{coincidencia:.3f}** (1 indica coincidencia perfecta). Identifica en las tablas cuáles categorías aparecen mezcladas.

**6. ¿Podrían mejorar los embeddings de documentos?**  
**Potencialmente sí.** Pueden captar relaciones semánticas entre documentos que usan palabras diferentes. No garantizan mejores clusters: depende del modelo, los datos y la evaluación.

### Conclusión
La vectorización modifica las distancias entre textos y, por tanto, puede cambiar los clusters aun usando el mismo K-means. Para este conjunto, contrastamos la silueta, los términos de los centroides y la composición temática para decidir qué representación resulta más útil.
"""

display(Markdown(texto))

---
**Antes de entregar:** comprueba las curvas de inercia y silueta, las listas de 10 palabras por cluster y las tablas de categorías. Ajusta las respuestas 4 y 5 con tus propias observaciones. Puedes entregar directamente este archivo `.ipynb`.

*Fuente del procedimiento: Sesión 19, «Aprendizaje de máquina para textos», diapositivas 38–48; enunciado de Práctica #6.*

# R3 visualization

## 10. Visualización interactiva en $\mathbb{R}^3$

Proyectaremos **TF-IDF** y **Count + L2** a tres dimensiones mediante **PCA**. Después normalizaremos *las coordenadas 3D* para verlas sobre una esfera y colorearemos los clusters de K-means.

**Nota:** K-means se entrenó anteriormente con los vectores completos, **no** con PCA. La esfera representa la normalización de la *proyección 3D*, que no es equivalente a normalizar los documentos en el espacio original.

Usamos una muestra fija de documentos para que los gráficos 3D sean fluidos.

In [ ]:
# Plotly permite rotar y acercar los gráficos con el ratón.
# Si hace falta: %pip install plotly
import plotly.graph_objects as go
from sklearn.decomposition import PCA

max_docs_r3 = 1000
n_vecinos = 15  # Documentos más cercanos a cada centroide

rng = np.random.default_rng(42)
indices_r3 = np.sort(rng.choice(
    len(documentos), size=min(max_docs_r3, len(documentos)), replace=False
))

print(f"Visualizaremos {len(indices_r3)} documentos (la misma muestra en ambos métodos).")

### PCA, normalización y vecinos del centroide

- **PCA:** comprime los documentos a tres coordenadas para visualizarlos.
- **Esfera:** aplica norma L2 a esas tres coordenadas, de modo que cada punto no nulo tenga longitud 1.
- **Vecinos:** selecciona los documentos más cercanos a **cada centroide de K-means** en el espacio vectorial original. No es un clasificador KNN.

In [ ]:
visualizaciones_r3 = {}

for nombre, X in matrices.items():
    modelo = modelos_elegidos[nombre]
    X_muestra = X[indices_r3].toarray().astype(np.float32)

    pca = PCA(n_components=3, svd_solver="randomized", random_state=42)
    coords = pca.fit_transform(X_muestra)
    esfera = normalize(coords, norm="l2")
    grupos = modelo.labels_[indices_r3]

    # Vecinos más cercanos a cada centroide según K-means original.
    distancias = modelo.transform(X[indices_r3])
    vecinos = {}
    for k in range(k_elegido):
        miembros = np.flatnonzero(grupos == k)
        orden = np.argsort(distancias[miembros, k])
        vecinos[k] = miembros[orden[:n_vecinos]]

    # Los centroides se proyectan con el mismo PCA de la muestra.
    centroides_3d = pca.transform(modelo.cluster_centers_)

    visualizaciones_r3[nombre] = dict(
        coords=coords, esfera=esfera, grupos=grupos,
        vecinos=vecinos, centroides=centroides_3d,
        varianza=pca.explained_variance_ratio_.sum(),
    )
    print(f"{nombre}: varianza explicada en 3D = {pca.explained_variance_ratio_.sum():.1%}")

# Comprobar las normas de las coordenadas 3D normalizadas.
for nombre, datos_r3 in visualizaciones_r3.items():
    normas = np.linalg.norm(datos_r3["esfera"], axis=1)
    print(f"{nombre} · puntos de norma ≈ 1:", np.isclose(normas[normas > 0], 1).all())

### Explorar en 3D

Usa los **botones** de cada figura: **PCA** → **Esfera** → **K-means** → **K-means + esfera** → **Vecinos**. Arrastra para rotar y usa la rueda para acercar.

En **Vecinos**, los puntos resaltados son los más próximos a su centroide **en las 5000 dimensiones originales** (o las que tenga la matriz). Una proyección 3D puede hacer que parezcan menos próximos.

In [ ]:
def mostrar_r3(nombre):
    datos_r3 = visualizaciones_r3[nombre]
    coords, esfera = datos_r3["coords"], datos_r3["esfera"]
    grupos, vecinos = datos_r3["grupos"], datos_r3["vecinos"]
    categorias_muestra = np.asarray(etiquetas_reales)[indices_r3]

    # Metadatos al pasar el cursor sobre un punto.
    info = np.column_stack((indices_r3, categorias_muestra))
    hover = "Documento: %{customdata[0]}<br>Categoría: %{customdata[1]}<extra>%{fullData.name}</extra>"
    colores = ["#2563eb", "#ea580c", "#16a34a", "#a855f7",
               "#e11d48", "#0891b2", "#b45309", "#64748b"]

    # Superficie tenue para mostrar la esfera de radio 1.
    phi = np.linspace(0, 2 * np.pi, 35)
    theta = np.linspace(0, np.pi, 20)
    sx = np.outer(np.cos(phi), np.sin(theta))
    sy = np.outer(np.sin(phi), np.sin(theta))
    sz = np.outer(np.ones_like(phi), np.cos(theta))

    fig = go.Figure()
    vistas = {}

    def agregar_puntos(puntos, indices, etiqueta, color, tamano=3, opacidad=0.60, leyenda=False):
        fig.add_trace(go.Scatter3d(
            x=puntos[indices, 0], y=puntos[indices, 1], z=puntos[indices, 2],
            mode="markers", name=etiqueta,
            marker=dict(size=tamano, color=color, opacity=opacidad),
            customdata=info[indices], hovertemplate=hover,
            showlegend=leyenda, visible=False,
        ))
        return len(fig.data) - 1

    def agregar_esfera():
        fig.add_trace(go.Surface(
            x=sx, y=sy, z=sz,
            colorscale=[[0, "#93c5fd"], [1, "#93c5fd"]],
            showscale=False, opacity=0.12, hoverinfo="skip", visible=False,
            showlegend=False, name="Esfera L2",
        ))
        return len(fig.data) - 1

    todos = np.arange(len(indices_r3))
    vistas["PCA"] = [agregar_puntos(coords, todos, "Documentos", "#64748b")]
    vistas["Esfera"] = [agregar_esfera(),
                        agregar_puntos(esfera, todos, "Documentos normalizados", "#64748b")]

    # Clusters proyectados en PCA (el modelo no se reentrena).
    trazas = []
    for k in range(k_elegido):
        idx = np.flatnonzero(grupos == k)
        trazas.append(agregar_puntos(coords, idx, f"Cluster {k}",
                                     colores[k % len(colores)], leyenda=True))
    fig.add_trace(go.Scatter3d(
        x=datos_r3["centroides"][:, 0],
        y=datos_r3["centroides"][:, 1],
        z=datos_r3["centroides"][:, 2],
        mode="markers+text", marker=dict(size=8, color="#111827", symbol="diamond"),
        text=[f"C{k}" for k in range(k_elegido)], textposition="top center",
        name="Centroides proyectados", showlegend=True, visible=False,
    ))
    trazas.append(len(fig.data) - 1)
    vistas["K-means"] = trazas

    # Los mismos clusters, pero con la proyección 3D normalizada.
    trazas = [agregar_esfera()]
    for k in range(k_elegido):
        idx = np.flatnonzero(grupos == k)
        trazas.append(agregar_puntos(esfera, idx, f"Cluster {k}",
                                     colores[k % len(colores)], leyenda=True))
    vistas["K-means + esfera"] = trazas

    # Resaltar los n_vecinos documentos cercanos al centroide de cada grupo.
    trazas = [agregar_esfera(),
              agregar_puntos(esfera, todos, "Otros documentos", "#cbd5e1", opacidad=0.13)]
    for k in range(k_elegido):
        trazas.append(agregar_puntos(esfera, vecinos[k], f"Vecinos · cluster {k}",
                                     colores[k % len(colores)], tamano=7,
                                     opacidad=0.95, leyenda=True))
    vistas["Vecinos"] = trazas

    botones = []
    for vista, ids in vistas.items():
        es_esfera = vista in ("Esfera", "K-means + esfera", "Vecinos")
        escena = dict(
            xaxis=dict(title="Componente 1", range=[-1.12, 1.12] if es_esfera else None),
            yaxis=dict(title="Componente 2", range=[-1.12, 1.12] if es_esfera else None),
            zaxis=dict(title="Componente 3", range=[-1.12, 1.12] if es_esfera else None),
            aspectmode="cube" if es_esfera else "data",
        )
        botones.append(dict(
            label=vista, method="update",
            args=[{"visible": [i in ids for i in range(len(fig.data))]},
                  {"title": f"{nombre} · {vista} (K={k_elegido})", "scene": escena}],
        ))

    # Comenzar con PCA sin etiquetas de cluster.
    for indice in vistas["PCA"]:
        fig.data[indice].visible = True

    fig.update_layout(
        title=f"{nombre} · PCA (K={k_elegido})",
        height=650,
        margin=dict(l=0, r=0, b=0, t=105),
        legend=dict(x=0.01, y=0.97, bgcolor="rgba(255,255,255,0.65)"),
        updatemenus=[dict(type="buttons", direction="right", buttons=botones,
                          x=0.01, y=1.13, xanchor="left", yanchor="top",
                          bgcolor="#f8fafc", showactive=True)],
        scene=dict(xaxis_title="Componente 1", yaxis_title="Componente 2",
                   zaxis_title="Componente 3", aspectmode="data"),
    )
    fig.show()
    return fig

# Una figura interactiva para cada representación.
fig_tfidf_r3 = mostrar_r3("TF-IDF")
fig_count_r3 = mostrar_r3("Count + L2")

### Qué observar

1. ¿Qué clusters se separan visualmente mejor en TF-IDF y Count + L2?
2. ¿Qué cambia al llevar la proyección 3D a la esfera?
3. ¿Los puntos destacados en **Vecinos** se concentran en una región de cada cluster?

**Interpretación:** la vista 3D es una aproximación. Las métricas, los centroides y las asignaciones de K-means se calcularon con la representación original de alta dimensión, no sobre la esfera.